# ⚙️ Análise de Sobrevivência de Motores Turbofan — CMAPSS FD001

## Contexto

A Análise de Sobrevivência é um conjunto de técnicas estatísticas desenvolvido originalmente na área médica para modelar o tempo até a ocorrência de um evento, tipicamente a morte de um paciente. Na engenharia de manutenção, esse "evento" é a falha do equipamento, e as mesmas ferramentas se aplicam com igual rigor para responder uma pergunta fundamental: *qual a probabilidade de um motor ainda estar operacional após N ciclos?*

Enquanto a análise exploratória anterior caracterizou o comportamento dos sensores ao longo da degradação, esta etapa foca na **distribuição do tempo até a falha** em si — sem depender de nenhuma variável preditora. Isso nos fornece um baseline estatístico robusto e uma compreensão probabilística da vida útil dos motores antes de qualquer modelo de machine learning.

Utilizamos o dataset CMAPSS FD001 da NASA, composto por 100 motores turbofan que operam sob condição única até a falha completa — um cenário ideal para análise de sobrevivência, onde todos os eventos são observados (ausência de censura).

## Perguntas que esta análise responde

**1. Qual a função de sobrevivência empírica dos motores?**  
O estimador de Kaplan-Meier constrói a curva de sobrevivência diretamente dos dados, sem assumir nenhuma distribuição, respondendo: qual a probabilidade de um motor sobreviver além de X ciclos?

**2. A taxa de falha dos motores aumenta com o tempo?**  
Motores mecânicos em degradação tipicamente apresentam taxa de falha crescente — o motor fica mais propenso a falhar a cada ciclo adicional. O estimador de Nelson-Aalen quantifica o risco acumulado diretamente, e a inclinação de sua curva revela se esse risco cresce, é constante ou decresce ao longo do tempo.

**3. Qual distribuição estatística melhor descreve a vida útil dos motores?**  
Testamos quatro distribuições paramétricas clássicas da literatura de confiabilidade — Weibull, Log-Normal, Log-Logística e Exponencial, e comparamos o ajuste de cada uma pelo Critério de Informação de Akaike (AIC), que penaliza modelos mais complexos para evitar overfitting.

**4. Como os modelos paramétricos se comparam entre si e com a curva empírica?**  
Sobrepondo as quatro distribuições ajustadas ao estimador de Kaplan-Meier, avaliamos visualmente qual modelo captura melhor o comportamento real dos motores, tanto na região de maior concentração de falhas quanto nas caudas da distribuição.

**5. Quais são os percentis de vida útil e o que eles significam na prática?**  
Extraímos os quantis de sobrevivência empíricos — em quantos ciclos 10%, 25%, 50%, 75% e 90% dos motores já teriam falhado, fornecendo referências diretas para definição de janelas de manutenção preventiva.

## 0. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter

plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.family'] = 'sans-serif'

## 1. Estimador de Kaplan-Meier e Risco Acumulado (Nelson-Aalen)

In [ ]:
COL_NAMES = [
    'unit_id', 'time_cycles',
    'op_setting_1', 'op_setting_2', 'op_setting_3',
    'sensor_temp_fan_inlet', 'sensor_temp_lpc_outlet', 'sensor_temp_hpc_outlet',
    'sensor_temp_lpt_outlet', 'sensor_pressure_inlet', 'sensor_pressure_fan_inlet',
    'sensor_pressure_ratio', 'sensor_physical_fan_speed', 'sensor_physical_core_speed',
    'sensor_engine_pressure_ratio', 'sensor_static_hpc_outlet', 'sensor_fuel_flow_ps30',
    'sensor_corrected_fan_speed', 'sensor_corrected_core_speed', 'sensor_bypass_ratio',
    'sensor_bleed_enthalpy', 'sensor_demanded_fan_speed', 'sensor_demanded_corrected_fan_speed',
    'sensor_hpt_coolant_bleed', 'sensor_lpt_coolant_bleed', 'sensor_bpt_ratio'
]

df_train = pd.read_csv('data/train_FD001.txt', sep='\s+', header=None, names=COL_NAMES)

# Vida útil de cada motor = ciclo máximo observado
durations = df_train.groupby('unit_id')['time_cycles'].max()

# Todos os motores falharam (sem censura no CMAPSS FD001)
event_observed = np.ones(len(durations))

print(f'Motores: {len(durations)}')
print(f'Vida útil média: {durations.mean():.1f} ciclos')
print(f'Mín: {durations.min()} | Máx: {durations.max()}')

![](midia/kaplan.png)

### Observações — Seção 1:

**Kaplan-Meier:**
A curva de sobrevivência revela um comportamento característico de componentes mecânicos em desgaste: nenhum motor falha antes do ciclo 128, sugerindo a existência de um **período de vida mínimo garantido** — os motores operam de forma estável até que o processo de degradação se torna crítico.

A queda é abrupta e concentrada entre os ciclos 150 e 250, onde a maioria das falhas ocorre. O intervalo de confiança se alarga progressivamente após o ciclo 250, reflexo da amostra reduzida de motores ainda em operação nessa faixa — os 3 outliers acima de 300 ciclos identificados na EDA.

A mediana de sobrevivência é de **199 ciclos**: metade dos motores falha antes desse ponto.

**Nelson-Aalen:**
O risco acumulado H(t) permanece praticamente nulo até o ciclo 130, confirmando o período de vida mínimo. A partir daí, a inclinação da curva cresce continuamente — evidência direta de que a **taxa de falha instantânea aumenta com o tempo**.

Esse comportamento, chamado de *desgaste progressivo*, é fisicamente esperado em motores aeronáuticos: cada ciclo adicional de operação sob condições severas aumenta a probabilidade de falha no ciclo seguinte.

Compreendida a curva de desgaste, vamos agora calcular agora as proporções de falha ao longo da vida útil da frota de motores.

## 2. Proporção Acumulada de Falhas


In [ ]:
from lifelines import KaplanMeierFitter

# CDF = 1 - S(t)
cdf = 1 - kmf.survival_function_

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(cdf.index, cdf['Motores FD001'] * 100, 
        color='steelblue', linewidth=2)
ax.fill_between(cdf.index, cdf['Motores FD001'] * 100, 
                alpha=0.15, color='steelblue')

# Marcadores nos percentis chave
for ciclo, pct in [(154, 10), (174, 25), (199, 50), (229, 75), (275, 90)]:
    ax.axvline(ciclo, color='darkred', linestyle='--', linewidth=1, alpha=0.6)
    ax.annotate(f'{pct}%\n({ciclo} ciclos)', 
                xy=(ciclo, pct), xytext=(ciclo+4, pct-6),
                fontsize=8, color='darkred', fontweight='bold')

ax.set_title('Proporção Acumulada de Falhas — CMAPSS FD001', fontweight='bold', fontsize=12)
ax.set_xlabel('Ciclos de Operação')
ax.set_ylabel('% de Motores que Falharam')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0f}%'))
ax.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig('failure_cdf.png', dpi=150, bbox_inches='tight')
plt.show()

![Proporção Acumulada de Falhas](midia/falhas.png)

### Observações — Seção 2

O gráfico de CDF complementa o Kaplan-Meier lendo a mesma informação de forma mais intuitiva — em vez de "probabilidade de sobreviver", mostra diretamente "quantos % já falharam".

Alguns pontos relevantes:

- Até o ciclo 154, apenas 10% dos motores falharam — a grande maioria ainda está operacional, confirmando o período de estabilidade inicial
- A concentração de falhas é intensa entre os ciclos 154 e 229, onde a proporção salta de 10% para 75% — **75% das falhas ocorrem num intervalo de apenas 75 ciclos**
- Após o ciclo 250, a curva desacelera visivelmente — os motores remanescentes são os outliers de vida longa identificados na EDA

**Aplicação prática:**
Esses percentis têm interpretação direta para políticas de manutenção preventiva. Uma estratégia conservadora interviria no ciclo 154 (P10) para garantir que apenas 10% dos motores cheguem à falha. Uma estratégia mais tolerante ao risco poderia usar o ciclo 199 (P50), aceitando que metade da frota falhe antes da manutenção.

A escolha do ponto de intervenção é um trade-off entre custo de manutenção desnecessária e custo de falha em operação — decisão que vai além da estatística e depende do contexto operacional.


Com o comportamento empírico bem caracterizado, ajustamos agora modelos paramétricos para representar matematicamente a distribuição de vida útil dos motores.

## 3. Comparação de distribuições de sobrevivência 


In [ ]:
from lifelines import WeibullFitter, LogNormalFitter, ExponentialFitter, LogLogisticFitter

# Ajustando os 4 modelos
modelos = {
    'Weibull':      WeibullFitter(),
    'Log-Normal':   LogNormalFitter(),
    'Exponencial':  ExponentialFitter(),
    'Log-Logística': LogLogisticFitter()
}

cores = {
    'Weibull':       'darkred',
    'Log-Normal':    'darkorange',
    'Exponencial':   'green',
    'Log-Logística': 'purple'
}

resultados = []

for nome, modelo in modelos.items():
    modelo.fit(durations, event_observed=event_observed, label=nome)
    resultados.append({
        'Modelo': nome,
        'AIC':    round(modelo.AIC_, 2),
        'Mediana (ciclos)': round(modelo.median_survival_time_, 1)
    })

df_aic = pd.DataFrame(resultados).sort_values('AIC').reset_index(drop=True)
df_aic['Δ AIC'] = (df_aic['AIC'] - df_aic['AIC'].min()).round(2)

# --- Plot: todos os modelos vs Kaplan-Meier ---
fig, ax = plt.subplots(figsize=(11, 6))

kmf.plot_survival_function(ax=ax, ci_show=True, color='steelblue',
                            linewidth=2.5, label='Kaplan-Meier (empírico)')

for nome, modelo in modelos.items():
    modelo.plot_survival_function(ax=ax, ci_show=False,
                                   color=cores[nome], linewidth=1.8,
                                   linestyle='--', label=nome)

ax.set_title('Comparação de Distribuições de Sobrevivência — CMAPSS FD001',
             fontweight='bold', fontsize=12)
ax.set_xlabel('Ciclos de Operação')
ax.set_ylabel('Probabilidade de Sobrevivência S(t)')
ax.legend(fontsize=9)
ax.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig('distribution_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# --- Tabela AIC ---
print('\n=== Comparação por AIC (menor = melhor ajuste) ===\n')
print(df_aic.to_string(index=False))

![Comparação de Distribuições](midia/comparacao.png)

| Modelo        |     AIC | Mediana (ciclos) |  Δ AIC |
| ------------- | ------: | ---------------: | -----: |
| Log-Normal    | 1038.91 |            201.6 |   0.00 |
| Log-Logística | 1039.49 |            199.8 |   0.58 |
| Weibull       | 1065.50 |            207.1 |  26.59 |
| Exponencial   | 1267.88 |            143.0 | 228.97 |


### Observações — Seção 3

**Visualmente**, Log-Normal e Log-Logística são as que mais se aproximam da curva empírica do Kaplan-Meier ao longo de toda a vida útil. O Weibull se desvia perceptivelmente no início da queda (ciclos 130–170), antecipando falhas que empiricamente ainda não ocorreram. A Exponencial é claramente inadequada — sua curva começa a cair desde o ciclo 0, ignorando completamente o período de estabilidade inicial dos motores.

**Pelo AIC**, a hierarquia é confirmada numericamente. Log-Normal vence com ΔAIC = 26.59 sobre o Weibull — diferença considerada evidência forte na literatura de que o processo de degradação deste turbofan segue um mecanismo **multiplicativo**: cada ciclo amplifica o dano acumulado em vez de simplesmente somá-lo. Em termos físicos, isso significa que o motor que já está degradado degrada *mais rápido* — o que é consistente com o padrão exponencial observado nos sensores na EDA.

Log-Normal e Log-Logística ficaram estatisticamente empatadas (ΔAIC = 0.58). Ambas descrevem bem o comportamento central, com a Log-Logística tendo leve vantagem na cauda direita ao capturar melhor os 3 motores de vida longa identificados anteriormente — mas a diferença é pequena demais para preferir uma sobre a outra com confiança.


## Conclusão Geral — Análise de Sobrevivência

Esta análise caracterizou probabilisticamente a vida útil dos 100 motores turbofan do CMAPSS FD001, estabelecendo um baseline estatístico robusto para as etapas de modelagem preditiva.

**Principais achados:**

**Comportamento empírico (Kaplan-Meier e Nelson-Aalen):**
Nenhum motor falha antes do ciclo 128, evidenciando um período de vida mínimo garantido em que os motores operam de forma estável. A partir desse ponto, a taxa de falha cresce continuamente — confirmada pela inclinação crescente do estimador de Nelson-Aalen — com concentração intensa de falhas entre os ciclos 150 e 250, onde 75% da frota é perdida em apenas 75 ciclos.

**Percentis de vida útil:**
Os quantis empíricos fornecem referências práticas para políticas de manutenção: intervir no ciclo 154 garante que apenas 10% dos motores cheguem à falha; usar o ciclo 199 (mediana) implica aceitar que metade da frota falhe antes da intervenção. A escolha do ponto de corte é um trade-off operacional entre custo de manutenção preventiva e custo de falha em serviço.

**Ajuste paramétrico:**
Contrariando a expectativa inicial — Weibull é o modelo padrão da literatura de confiabilidade — a **Log-Normal apresentou o melhor ajuste** (ΔAIC = 26.59 sobre o Weibull), seguida de perto pela Log-Logística (ΔAIC = 0.58). Esse resultado sugere que a degradação do turbofan segue um mecanismo multiplicativo: motores já degradados degradam mais rápido, o que é fisicamente consistente com o padrão exponencial de aceleração nos sensores observado na análise exploratória. A distribuição Exponencial foi descartada — sua premissa de taxa de falha constante é incompatível com o comportamento real dos motores.

**Próximo notebook → Feature Engineering e Modelagem Preditiva**

Com o comportamento estatístico da falha bem caracterizado, a próxima etapa constrói as features preditivas — normalização por motor, RUL cap, rolling statistics — e treina os modelos de estimação de Vida Útil Remanescente.